# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Load the starter data (same file used in Week 1).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows, {df.shape[1]} columns")

30,000 rows, 44 columns


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Content refresh prioritization.**

This is a **scoring / ranking** task, not a plain classification task. The content team doesn't need a yes/no verdict on every page — they need an ordered queue: given a fixed weekly review capacity (~50 pages), which pages should be looked at *first*? The underlying model still learns a binary pattern (is this page declining?), but the model's output — a continuous risk score — is used to **rank** pages, and the thing that matters is the quality of the top of that ranking, not overall classification accuracy. A page correctly labeled 'not declining' at rank #4,000 has zero value to the team; a correctly-ranked page in the top 50 does. That's what makes this ranking/scoring rather than plain classification.

In [2]:
# Sanity check: this is a ranking problem because capacity is fixed and small
# relative to the population — the model's job is to sort, not just label.
weekly_capacity = 50
print(f"{len(df):,} pages total vs a review capacity of {weekly_capacity} per week")
print(f"That's {weekly_capacity/len(df)*100:.2f}% of pages reviewable per week -> ranking quality at the top matters most")

30,000 pages total vs a review capacity of 50 per week
That's 0.17% of pages reviewable per week -> ranking quality at the top matters most


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining` = 1 if `trend_direction == "down"`, else 0.**

`trend_direction` is computed from `trend_pct`, which compares each page's traffic in the **last 30 days** against its **previous 30 days** (`impressions_last_30d` / `clicks_last_30d` vs `impressions_prev_30d` / `clicks_prev_30d`). That means the label is an **observed outcome** — a real before/after comparison in the traffic data — not a hand-written rule I'm defining myself. That matters: if I invented the rule, a model trained on it would just learn to reproduce my rule, not learn anything new about which pages actually decline. Using the measured trend as the label means the model has to find real patterns in the input signals (position, CTR, engagement, freshness, content type) that predict an outcome that already happened in the data, independent of my opinion.

In [3]:
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(f"Declining rate: {df['is_declining'].mean():.1%} of pages ({df['is_declining'].sum():,} of {len(df):,})")

# confirm the label really does come from a measured before/after comparison, not a static field
print(df[["impressions_prev_30d", "impressions_last_30d", "trend_pct", "trend_direction"]].head(3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2% of pages (16,262 of 30,000)
   impressions_prev_30d  impressions_last_30d  trend_pct trend_direction
0                   987                   578      -41.4            down
1                  5915                  2501      -57.7            down
2                  6089                  2382      -60.9            down


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** — of the top 50 pages the model ranks highest, what fraction are actually declining (`is_declining == 1`)?

I picked this over accuracy or plain ROC-AUC because the team never looks past their weekly capacity — pages ranked 51st onward get zero attention regardless of how the model scored them. A model with great overall accuracy but a mediocre top-50 is useless here; a model with so-so accuracy everywhere else but a sharp top-50 is exactly what's needed. Precision@50 also has an immediate plain-English reading a non-technical stakeholder can check by hand: 'of this week's 50 review picks, how many turned out to actually be declining?'

'Good' means beating the base rate (54%, i.e. what a random 50 pages would give you) by a wide, defensible margin — not just beating 0.

In [4]:
def precision_at_k(labels, scores, k):
    order = scores.sort_values(ascending=False).index[:k]
    return labels.loc[order].mean()

# a naive one-signal baseline: rank purely by search demand (search_volume)
naive_precision_50 = precision_at_k(df["is_declining"], df["search_volume"], 50)
base_rate = df["is_declining"].mean()
print(f"Naive 'rank by search_volume' baseline, Precision@50: {naive_precision_50:.2f}")
print(f"Base rate (a random 50 pages would score about this): {base_rate:.2f}")
print("search_volume alone actually ranks worse than random here -- demand size doesn't predict decline.")
print("This is the bar my Week 4+ model needs to clear convincingly.")

Naive 'rank by search_volume' baseline, Precision@50: 0.44
Base rate (a random 50 pages would score about this): 0.54
search_volume alone actually ranks worse than random here -- demand size doesn't predict decline.
This is the bar my Week 4+ model needs to clear convincingly.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one published content page belonging to one client** — identified by `content_id` (nested under `client_id`). Each row carries that page's search-demand signals (`search_volume`, `competition`), on-page signals (`content_type`, `word_count`, `content_age_days`, `days_since_last_update`), performance signals (`avg_position`, `ctr`, `impressions_90d`, `engagement_rate`), and the observed outcome (`trend_direction`, `trend_pct`) the target is built from. This is the natural unit because it's also the unit the content team acts on — a reviewer picks up one page at a time, not a whole client's site.

In [5]:
unit_of_analysis_cols = [
    "content_id", "client_id", "content_type", "word_count",
    "content_age_days", "days_since_last_update", "avg_position", "ctr",
    "impressions_90d", "engagement_rate", "trend_pct", "trend_direction", "is_declining",
]
df[unit_of_analysis_cols].head(5)

,content_id,client_id,content_type,word_count,content_age_days,days_since_last_update,avg_position,ctr,impressions_90d,engagement_rate,trend_pct,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,187,20,10.6,0.76,3803,5.88,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,445,25,20.3,0.05,15320,0.00,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,141,20,36.5,0.09,12581,0.00,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,463,22,6.2,0.49,11751,1.28,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,263,14,44.0,0.13,19140,0.00,-34.7,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Decline isn't driven by one signal — it's a tangled mix of position drift, CTR erosion, content staleness, and demand shifts, and which signal matters shifts from page to page. A hand-written rule has to pick 1-2 thresholds (e.g. 'flag anything older than 300 days') and misses everything that decays for a different reason. The numbers below show this isn't a clean single-variable pattern: content age and update recency barely separate decliners from the rest on their own, and simply ranking by search demand does no better than chance (Precision@50 = 0.44, below the 0.54 base rate). Week 1's pipeline showed the same story from the other direction: a more careful hand-written multi-condition rule still only reached Precision@50 = 0.24 on held-out data, while a model that combines many weak, correlated signals reached 0.74. Neither a single field nor a person's best-guess rule gets close — the signal only emerges once many weak, tangled features are combined, which is exactly what a fixed if-statement can't do and a learned model can.

In [6]:
# Evidence the pattern is messy, not a clean single-variable split
print("Median content_age_days by trend_direction (barely separates 'down' from the rest):")
print(df.groupby("trend_direction")["content_age_days"].median().round(0))

print()
print("Median days_since_last_update by trend_direction (also weak on its own):")
print(df.groupby("trend_direction")["days_since_last_update"].median().round(0))

print()
print(f"My naive single-signal baseline (rank by search_volume), Precision@50: {naive_precision_50:.2f}")
print("Week 1's hand-written multi-condition rule (held-out eval), Precision@50: 0.24")
print("Week 1's trained random forest (many signals combined), Precision@50: 0.74")
print("No single field, and no hand-written rule, gets close -> the signal needs the combination.")

Median content_age_days by trend_direction (barely separates 'down' from the rest):
trend_direction
down      216.0
flat      231.0
new       279.0
stable    300.0
up        292.0
Name: content_age_days, dtype: float64

Median days_since_last_update by trend_direction (also weak on its own):
trend_direction
down      20.0
flat      20.0
new       20.0
stable    22.0
up        22.0
Name: days_since_last_update, dtype: float64

My naive single-signal baseline (rank by search_volume), Precision@50: 0.44
Week 1's hand-written multi-condition rule (held-out eval), Precision@50: 0.24
Week 1's trained random forest (many signals combined), Precision@50: 0.74
No single field, and no hand-written rule, gets close -> the signal needs the combination.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.